<a href="https://colab.research.google.com/github/Adamphoenix003/GNN-LinkPrediction/blob/main/SFDDGNNversion3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
import subprocess, sys

def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", *args, "-q"])

# ── Step 1: upgrade numpy to satisfy Colab's other packages ──
pip("numpy>=2.0")

# ── Step 2: install PyTorch Geometric (pure-Python wheels, ──
#           no CUDA-specific torch-scatter/torch-sparse needed
#           for the ops we use in Pipeline 1)
pip("torch-geometric")

# ── Step 3: node2vec (pure Python, no native deps) ───────────
pip("node2vec")

# ── Step 4: scikit-learn (usually already in Colab) ──────────
pip("scikit-learn")

# ── Step 5: scipy (used for sparse adjacency matrix) ─────────
pip("scipy")

# ── Step 6: gensim (Word2Vec backend used by node2vec) ───────
pip("gensim")

print("\n✅ All dependencies installed.")
print("👉 Now go to  Runtime → Restart Runtime  and then run Cell 2 onwards.")


✅ All dependencies installed.
👉 Now go to  Runtime → Restart Runtime  and then run Cell 2 onwards.


In [1]:

import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam

import torch_geometric
from torch_geometric.datasets import Planetoid
from torch_geometric.transforms import RandomLinkSplit
from torch_geometric.utils import (
    to_networkx, add_self_loops, negative_sampling,
    to_undirected, k_hop_subgraph
)
from torch_geometric.data import Data, DataLoader
from torch_geometric.nn import SAGEConv

from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.preprocessing import normalize

import networkx as nx
import warnings
warnings.filterwarnings("ignore")

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Imports done  |  device = {DEVICE}")
print(f"   torch-geometric {torch_geometric.__version__}")

/usr/local/lib/python3.12/dist-packages/torch_geometric/__init__.py:4: UserWarning: An issue occurred while importing 'torch-scatter'. Disabling its usage. Stacktrace: Could not load this library: /usr/local/lib/python3.12/dist-packages/torch_scatter/_version_cuda.so
  import torch_geometric.typing
/usr/local/lib/python3.12/dist-packages/torch_geometric/__init__.py:4: UserWarning: An issue occurred while importing 'torch-sparse'. Disabling its usage. Stacktrace: Could not load this library: /usr/local/lib/python3.12/dist-packages/torch_sparse/_version_cuda.so
  import torch_geometric.typing


✅ Imports done  |  device = cuda
   torch-geometric 2.7.0


In [2]:
ROOT = "./data"
dataset = Planetoid(root=ROOT, name="Cora")
data    = dataset[0]

print(f"Cora statistics")
print(f"  nodes   : {data.num_nodes}")
print(f"  edges   : {data.num_edges // 2}  (undirected)")
print(f"  features: {data.num_node_features}")

# ── train/val/test split for LINK PREDICTION (80/10/10) ──────
transform = RandomLinkSplit(
    num_val          = 0.10,
    num_test         = 0.10,
    is_undirected    = True,
    add_negative_train_samples = True,   # adds equal neg edges for training
    neg_sampling_ratio         = 1.0,
)
train_data, val_data, test_data = transform(data)

print(f"\nLink-split summary")
print(f"  train pos edges : {train_data.edge_label_index.shape[1] // 2}")
print(f"  val   pos edges : {val_data.edge_label_index.shape[1]   // 2}")
print(f"  test  pos edges : {test_data.edge_label_index.shape[1]  // 2}")


Cora statistics
  nodes   : 2708
  edges   : 5278  (undirected)
  features: 1433

Link-split summary
  train pos edges : 4224
  val   pos edges : 527
  test  pos edges : 527


In [ ]:
#  The paper uses DeepWalk, Node2Vec, and Matrix Factorisation (MF)
#  to generate topology-only initial embeddings for each node.
#  Their outputs are concatenated and projected through a 2-layer MLP.

EMB_DIM  = 64    # embedding dim per basis method  (paper uses 128 for final)
FINAL_DIM = 128  # dimension of ELSE output fed into GraphSAGE

# ── 4a. Matrix Factorisation (SVD on adjacency matrix) ───────

def compute_mf_embedding(edge_index, num_nodes, dim=EMB_DIM):
    """
    Approximate SVD of the adjacency matrix.
    Returns a (num_nodes, dim) numpy array.
    """
    from scipy.sparse import coo_matrix
    src, dst = edge_index.cpu().numpy()
    # build undirected adjacency
    rows = np.concatenate([src, dst])
    cols = np.concatenate([dst, src])
    vals = np.ones(len(rows), dtype=np.float32)
    A = coo_matrix((vals, (rows, cols)), shape=(num_nodes, num_nodes))
    svd = TruncatedSVD(n_components=dim, random_state=SEED)
    emb = svd.fit_transform(A)
    return normalize(emb).astype(np.float32)

# ── 4b. DeepWalk embedding ────────────────────────────────────

def compute_deepwalk_embedding(edge_index, num_nodes, dim=EMB_DIM):
    """
    DeepWalk via random walks + Word2Vec (Skip-gram).
    Uses the node2vec library which supports both DeepWalk and Node2Vec.
    DeepWalk = Node2Vec with p=1, q=1 (unbiased walk).
    """
    from node2vec import Node2Vec as N2V

    # build networkx graph from edge_index
    G = nx.Graph()
    G.add_nodes_from(range(num_nodes))
    edges = edge_index.t().cpu().numpy().tolist()
    G.add_edges_from(edges)

    # remove isolated nodes for walk (node2vec requirement)
    G.remove_nodes_from(list(nx.isolates(G)))

    dw = N2V(
        G,
        dimensions  = dim,
        walk_length = 20,
        num_walks   = 10,
        p           = 1,   # DeepWalk: unbiased
        q           = 1,
        workers     = 1,
        seed        = SEED,
    )
    model = dw.fit(window=5, min_count=1, batch_words=4)

    # build embedding matrix; isolated nodes get zero vector
    emb = np.zeros((num_nodes, dim), dtype=np.float32)
    for node in range(num_nodes):
        if str(node) in model.wv:
            emb[node] = model.wv[str(node)]
    return normalize(emb).astype(np.float32)

# ── 4c. Node2Vec embedding ────────────────────────────────────

def compute_node2vec_embedding(edge_index, num_nodes, dim=EMB_DIM):
    """
    Node2Vec with p=0.25, q=4 (biased walk favouring BFS-style exploration).
    """
    from node2vec import Node2Vec as N2V

    G = nx.Graph()
    G.add_nodes_from(range(num_nodes))
    edges = edge_index.t().cpu().numpy().tolist()
    G.add_edges_from(edges)
    G.remove_nodes_from(list(nx.isolates(G)))

    n2v = N2V(
        G,
        dimensions  = dim,
        walk_length = 20,
        num_walks   = 10,
        p           = 0.25,
        q           = 4,
        workers     = 1,
        seed        = SEED,
    )
    model = n2v.fit(window=5, min_count=1, batch_words=4)

    emb = np.zeros((num_nodes, dim), dtype=np.float32)
    for node in range(num_nodes):
        if str(node) in model.wv:
            emb[node] = model.wv[str(node)]
    return normalize(emb).astype(np.float32)

# ── 4d. Train all three basis methods & build ELSE output ────

print("Training ELSE basis methods (this takes ~2-3 min on CPU)…")

edge_index_train = train_data.edge_index
num_nodes        = data.num_nodes

print("  [1/3] Matrix Factorisation…")
mf_emb  = compute_mf_embedding(edge_index_train, num_nodes)

print("  [2/3] DeepWalk…")
dw_emb  = compute_deepwalk_embedding(edge_index_train, num_nodes)

print("  [3/3] Node2Vec…")
n2v_emb = compute_node2vec_embedding(edge_index_train, num_nodes)

# Concatenate: shape = (num_nodes, 3 * EMB_DIM)
basis_emb = np.concatenate([mf_emb, dw_emb, n2v_emb], axis=1)
basis_tensor = torch.from_numpy(basis_emb).to(DEVICE)
print(f"✅ ELSE basis embeddings ready  |  shape = {basis_tensor.shape}")


In [4]:
# ── CELL 5 ── ELSE MLP: fuse basis embeddings → X_B ─────────
#
#  "MLP(∥_B B(A,X))" from the paper.
#  Input  : (num_nodes, 3 * EMB_DIM)
#  Output : (num_nodes, FINAL_DIM)  = X_B used as initial embedding for GNN

class ELSE_MLP(nn.Module):
    """
    2-layer MLP that projects concatenated basis embeddings
    into a single topology-aware initial embedding.
    """
    def __init__(self, in_dim, hidden_dim, out_dim, dropout=0.5):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, out_dim),
            nn.ReLU(),
        )

    def forward(self, x):
        return self.net(x)

else_mlp = ELSE_MLP(
    in_dim     = 3 * EMB_DIM,   # 192
    hidden_dim = 256,
    out_dim    = FINAL_DIM,      # 128
    dropout    = 0.5,
).to(DEVICE)
print(f"✅ ELSE MLP  |  params = {sum(p.numel() for p in else_mlp.parameters()):,}")






✅ ELSE MLP  |  params = 82,304


In [5]:
# ── CELL 6 ── GraphSAGE-Based Structure Encoder ──────────────
#
#  Key differences from vanilla GraphSAGE (Section 3.1.2):
#  1. After each message-passing layer, the EDGE embedding is
#     updated explicitly (residual connection on edge repr).
#  2. An edge-based binary cross-entropy loss jointly trains
#     the node aggregator (W_N) and the ReadOut function (W_E).

class GraphSAGE_StructureEncoder(nn.Module):
    """
    2-layer GraphSAGE with edge embedding update via residual connections.

    Node update  (eq. 3):  h^k_i  ← σ( W_N · Aggregator(h^{k-1}_i ∪ h^{k-1}_u) )
    Edge update  (eq. 4):  h^k_ij ← σ( W_E · Aggregator(h^{k-1}_ij, h^k_i, h^k_j) )
    ReadOut      (eq. 5):  p_ij   = δ( h_ij )   where δ = Sigmoid
    """

    def __init__(self, in_dim, hidden_dim, out_dim, dropout=0.5):
        super().__init__()

        # Node message-passing layers  (W_N)
        self.sage1 = SAGEConv(in_dim,     hidden_dim)
        self.sage2 = SAGEConv(hidden_dim, out_dim)

        # Edge update layers  (W_E) ── eq. 4
        # Input = concat(h^{k-1}_ij,  h^k_i,  h^k_j)
        self.edge_update1 = nn.Linear(in_dim * 2 + hidden_dim * 2, hidden_dim)
        self.edge_update2 = nn.Linear(hidden_dim + out_dim * 2,    out_dim)

        # ReadOut  (Sigmoid-based binary classifier on edge embedding)
        self.readout = nn.Sequential(
            nn.Linear(out_dim, out_dim // 2),
            nn.ReLU(),
            nn.Linear(out_dim // 2, 1),
        )

        self.dropout = nn.Dropout(dropout)
        self.out_dim = out_dim

    def forward(self, x, edge_index, edge_pairs):
        """
        x          : (N, in_dim)   initial node embeddings from ELSE
        edge_index : (2, E)        training graph edges
        edge_pairs : (2, P)        target node pairs to score
        Returns    : logits (P,)
        """
        src_pairs, dst_pairs = edge_pairs[0], edge_pairs[1]

        # ── Layer 1: node update ─────────────────────────────
        h0 = x                                           # initial node emb
        h1 = F.relu(self.sage1(h0, edge_index))
        h1 = self.dropout(h1)

        # ── Edge update after layer 1 (residual) ─────────────
        # Initial edge emb h^0_ij = σ(x_i || x_j)
        edge_h0 = torch.cat([h0[src_pairs], h0[dst_pairs]], dim=-1)   # (P, 2*in_dim)
        # h^1_ij ← σ( W_E · (h^0_ij || h^1_i || h^1_j) )
        edge_h1_input = torch.cat([edge_h0,
                                   h1[src_pairs],
                                   h1[dst_pairs]], dim=-1)             # (P, 2*in_dim + 2*hid)
        edge_h1 = F.relu(self.edge_update1(edge_h1_input))

        # ── Layer 2: node update ─────────────────────────────
        h2 = F.relu(self.sage2(h1, edge_index))
        h2 = self.dropout(h2)

        # ── Edge update after layer 2 (residual) ─────────────
        edge_h2_input = torch.cat([edge_h1,
                                   h2[src_pairs],
                                   h2[dst_pairs]], dim=-1)             # (P, hid + 2*out)
        edge_h2 = F.relu(self.edge_update2(edge_h2_input))

        # ── ReadOut: p_ij = δ(h_ij) ──────────────────────────
        logits = self.readout(edge_h2).squeeze(-1)                     # (P,)
        return logits, h2   # also return node embeddings for later use

    def get_node_embeddings(self, x, edge_index):
        """Pure-inference pass: returns final node-level H_S."""
        h1 = F.relu(self.sage1(x, edge_index))
        h2 = F.relu(self.sage2(h1, edge_index))
        return h2   # (N, out_dim)


structure_encoder = GraphSAGE_StructureEncoder(
    in_dim     = FINAL_DIM,   # 128  (output of ELSE MLP)
    hidden_dim = 256,
    out_dim    = 128,
    dropout    = 0.5,
).to(DEVICE)

print(f"✅ GraphSAGE Structure Encoder  |  params = "
      f"{sum(p.numel() for p in structure_encoder.parameters()):,}")

✅ GraphSAGE Structure Encoder  |  params = 402,305


In [6]:
# ── CELL 7 ── Training Pipeline 1 ────────────────────────────
#
#  Loss (eq. 5): L_S = -y·log(δ(h_ij)) - (1-y)·log(1-δ(h_ij))
#  Optimiser   : Adam, lr=1e-3, weight_decay=1e-5 (L2 reg)
#  Epochs      : early-stop on val AP (patience=50)

# ── Build static ELSE embedding (no gradients needed here) ───
else_mlp.eval()
with torch.no_grad():
    X_B = else_mlp(basis_tensor)          # (N, 128)  = X^B from paper

print(f"X_B (ELSE output) shape: {X_B.shape}")

# ── Hyperparameters ───────────────────────────────────────────
LR          = 1e-3
WEIGHT_DECAY= 1e-5
MAX_EPOCHS  = 300
PATIENCE    = 50
BATCH_SIZE  = 1024

# ── Optimisers ────────────────────────────────────────────────
# We jointly optimise ELSE MLP + Structure Encoder
optimizer = Adam(
    list(else_mlp.parameters()) + list(structure_encoder.parameters()),
    lr=LR, weight_decay=WEIGHT_DECAY
)

# ── Helper: build batched edge pairs + labels ─────────────────

def get_edges_and_labels(split_data, device):
    """
    Returns:
      edge_pairs : (2, 2P) – pos + neg pairs
      labels     : (2P,)   – 1 for positive, 0 for negative
    """
    pairs  = split_data.edge_label_index.to(device)  # (2, 2P)
    labels = split_data.edge_label.float().to(device) # (2P,)
    return pairs, labels

def evaluate(encoder, mlp, x_basis, split_data, edge_index_train, device):
    encoder.eval(); mlp.eval()
    with torch.no_grad():
        x_b = mlp(x_basis)
        pairs, labels = get_edges_and_labels(split_data, device)
        logits, _ = encoder(x_b, edge_index_train.to(device), pairs)
        probs = torch.sigmoid(logits).cpu().numpy()
        y     = labels.cpu().numpy()
    ap  = average_precision_score(y, probs)
    auc = roc_auc_score(y, probs)
    return ap, auc

# ── Training loop ─────────────────────────────────────────────
print(f"\nTraining Pipeline 1 (max {MAX_EPOCHS} epochs, patience={PATIENCE})…\n")

best_val_ap   = 0.0
patience_cnt  = 0
best_ckpt     = None

train_pairs, train_labels = get_edges_and_labels(train_data, DEVICE)
edge_index_tr = train_data.edge_index.to(DEVICE)

for epoch in range(1, MAX_EPOCHS + 1):
    else_mlp.train(); structure_encoder.train()

    # ── mini-batch loop over edge pairs ──────────────────────
    perm   = torch.randperm(train_pairs.shape[1])
    total_loss = 0.0
    num_batches = 0

    for start in range(0, train_pairs.shape[1], BATCH_SIZE):
        idx   = perm[start : start + BATCH_SIZE]
        batch_pairs  = train_pairs[:, idx]   # (2, B)
        batch_labels = train_labels[idx]     # (B,)

        optimizer.zero_grad()

        # Forward: ELSE MLP → X_B, then Structure Encoder
        x_b    = else_mlp(basis_tensor)
        logits, _ = structure_encoder(x_b, edge_index_tr, batch_pairs)

        # Edge-based binary cross-entropy (eq. 5)
        loss = F.binary_cross_entropy_with_logits(logits, batch_labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            list(else_mlp.parameters()) + list(structure_encoder.parameters()),
            max_norm=1.0
        )
        optimizer.step()

        total_loss += loss.item()
        num_batches += 1

    avg_loss = total_loss / max(num_batches, 1)

    # ── Validation every 5 epochs ────────────────────────────
    if epoch % 5 == 0:
        val_ap, val_auc = evaluate(
            structure_encoder, else_mlp, basis_tensor,
            val_data, edge_index_tr, DEVICE
        )
        print(f"  Epoch {epoch:4d} | loss={avg_loss:.4f} | "
              f"val AP={val_ap*100:.2f}%  AUC={val_auc*100:.2f}%")

        if val_ap > best_val_ap:
            best_val_ap  = val_ap
            patience_cnt = 0
            best_ckpt = {
                "else_mlp":          {k: v.clone() for k, v in else_mlp.state_dict().items()},
                "structure_encoder": {k: v.clone() for k, v in structure_encoder.state_dict().items()},
            }
        else:
            patience_cnt += 5
            if patience_cnt >= PATIENCE:
                print(f"\n  Early stop at epoch {epoch}  (best val AP={best_val_ap*100:.2f}%)")
                break

# ── Reload best checkpoint ────────────────────────────────────
if best_ckpt:
    else_mlp.load_state_dict(best_ckpt["else_mlp"])
    structure_encoder.load_state_dict(best_ckpt["structure_encoder"])

print("\n✅ Pipeline 1 training complete")




X_B (ELSE output) shape: torch.Size([2708, 128])

Training Pipeline 1 (max 300 epochs, patience=50)…

  Epoch    5 | loss=0.6421 | val AP=61.70%  AUC=57.43%
  Epoch   10 | loss=0.5512 | val AP=64.30%  AUC=59.28%
  Epoch   15 | loss=0.3577 | val AP=67.66%  AUC=65.52%
  Epoch   20 | loss=0.2859 | val AP=71.39%  AUC=69.24%
  Epoch   25 | loss=0.2399 | val AP=72.51%  AUC=70.69%
  Epoch   30 | loss=0.2183 | val AP=73.31%  AUC=71.45%
  Epoch   35 | loss=0.1901 | val AP=74.07%  AUC=72.12%
  Epoch   40 | loss=0.1737 | val AP=74.96%  AUC=72.98%
  Epoch   45 | loss=0.1534 | val AP=75.42%  AUC=73.47%
  Epoch   50 | loss=0.1283 | val AP=75.30%  AUC=73.79%
  Epoch   55 | loss=0.1123 | val AP=75.53%  AUC=73.83%
  Epoch   60 | loss=0.1012 | val AP=76.01%  AUC=74.51%
  Epoch   65 | loss=0.0874 | val AP=76.26%  AUC=74.97%
  Epoch   70 | loss=0.0689 | val AP=77.26%  AUC=75.54%
  Epoch   75 | loss=0.0633 | val AP=77.54%  AUC=75.64%
  Epoch   80 | loss=0.0543 | val AP=78.28%  AUC=76.40%
  Epoch   85 | los

In [7]:
# ── CELL 8 ── Evaluate on Test Set ───────────────────────────

test_ap, test_auc = evaluate(
    structure_encoder, else_mlp, basis_tensor,
    test_data, edge_index_tr, DEVICE
)

print(f"\n{'='*50}")
print(f"  Pipeline 1 (Structure-Only) – Test Results")
print(f"{'='*50}")
print(f"  AP  : {test_ap  * 100:.2f}%")
print(f"  AUC : {test_auc * 100:.2f}%")
print(f"{'='*50}")

# Paper target on Cora (Stru-Only Emb ablation): slightly below SFDDGNN
# but significantly above vanilla GNN baselines.


  Pipeline 1 (Structure-Only) – Test Results
  AP  : 82.35%
  AUC : 79.51%


In [8]:
# ── CELL 9 ── Extract & Save Structure-Only Embedding H_S ────
#
#  H_S is the output that feeds Pipeline 4 (Fusion).
#  We compute it here over ALL nodes using the full training graph.

else_mlp.eval(); structure_encoder.eval()
with torch.no_grad():
    X_B_final = else_mlp(basis_tensor)
    H_S = structure_encoder.get_node_embeddings(
        X_B_final, edge_index_tr
    )  # (N, 128)

print(f"H_S shape: {H_S.shape}  |  dtype: {H_S.dtype}")

# Save for later pipelines
torch.save({
    "H_S":                H_S.cpu(),
    "else_state":         else_mlp.state_dict(),
    "encoder_state":      structure_encoder.state_dict(),
    "basis_tensor":       basis_tensor.cpu(),
    "train_edge_index":   edge_index_tr.cpu(),
    "val_data":           val_data,
    "test_data":          test_data,
    "train_data":         train_data,
    "num_nodes":          num_nodes,
}, "pipeline1_outputs.pt")

print("✅ H_S saved to pipeline1_outputs.pt")
print("   Pass this file into Pipeline 2 → Pipeline 4.")


# ── CELL 10 ── Quick sanity checks & visualisation ───────────

print("\n── Sanity checks ──────────────────────────────────────")
print(f"  H_S min / max / mean : "
      f"{H_S.min().item():.4f} / {H_S.max().item():.4f} / {H_S.mean().item():.4f}")
print(f"  H_S contains NaN     : {torch.isnan(H_S).any().item()}")
print(f"  H_S contains Inf     : {torch.isinf(H_S).any().item()}")

# Cosine similarity between two known-connected nodes
# (pick first edge from test positive set)
pos_mask = test_data.edge_label.bool()
pos_pairs = test_data.edge_label_index[:, pos_mask]
i, j = pos_pairs[0, 0].item(), pos_pairs[1, 0].item()
sim_pos = F.cosine_similarity(H_S[i].unsqueeze(0), H_S[j].unsqueeze(0)).item()

# Pick a clearly unconnected pair (first negative edge)
neg_mask = ~pos_mask
neg_pairs = test_data.edge_label_index[:, neg_mask]
k, l = neg_pairs[0, 0].item(), neg_pairs[1, 0].item()
sim_neg = F.cosine_similarity(H_S[k].unsqueeze(0), H_S[l].unsqueeze(0)).item()

print(f"\n  Cosine sim (connected pair    {i}–{j}) : {sim_pos:.4f}")
print(f"  Cosine sim (unconnected pair  {k}–{l}) : {sim_neg:.4f}")
print(f"  Δ = {sim_pos - sim_neg:.4f}  (positive → model separates classes)\n")

print("=" * 50)
print("  Pipeline 1 complete. Next step → Pipeline 2")
print("  (Feature-Based Graph Representation Learning)")
print("=" * 50)

H_S shape: torch.Size([2708, 128])  |  dtype: torch.float32
✅ H_S saved to pipeline1_outputs.pt
   Pass this file into Pipeline 2 → Pipeline 4.

── Sanity checks ──────────────────────────────────────
  H_S min / max / mean : 0.0000 / 1.1844 / 0.0240
  H_S contains NaN     : False
  H_S contains Inf     : False

  Cosine sim (connected pair    306–910) : 0.1044
  Cosine sim (unconnected pair  1805–309) : 0.8497
  Δ = -0.7452  (positive → model separates classes)

  Pipeline 1 complete. Next step → Pipeline 2
  (Feature-Based Graph Representation Learning)


PIPELINE 2- FEATURE


In [9]:
import subprocess, sys

def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", *args, "-q"])

pip("transformers")          # HuggingFace transformers (RoBERTa)
pip("torch-geometric")       # should already be installed
pip("scikit-learn")
pip("scipy")
pip("numpy>=2.0")

print("✅ Pipeline 2 dependencies installed.")
print("👉 Runtime → Restart Runtime, then run Cell 2 onwards.")


✅ Pipeline 2 dependencies installed.
👉 Runtime → Restart Runtime, then run Cell 2 onwards.


In [1]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from torch.optim.lr_scheduler import StepLR
from torch.utils.data import Dataset, DataLoader

from transformers import RobertaTokenizer, RobertaModel

from torch_geometric.datasets import Planetoid
from torch_geometric.transforms import RandomLinkSplit
from torch_geometric.utils import to_networkx

from sklearn.metrics import average_precision_score, roc_auc_score

import networkx as nx
import warnings
warnings.filterwarnings("ignore")

# ── Reproducibility ──────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Imports done  |  device = {DEVICE}")


/usr/local/lib/python3.12/dist-packages/torch_geometric/__init__.py:4: UserWarning: An issue occurred while importing 'torch-scatter'. Disabling its usage. Stacktrace: Could not load this library: /usr/local/lib/python3.12/dist-packages/torch_scatter/_version_cuda.so
  import torch_geometric.typing
/usr/local/lib/python3.12/dist-packages/torch_geometric/__init__.py:4: UserWarning: An issue occurred while importing 'torch-sparse'. Disabling its usage. Stacktrace: Could not load this library: /usr/local/lib/python3.12/dist-packages/torch_sparse/_version_cuda.so
  import torch_geometric.typing


✅ Imports done  |  device = cuda


In [15]:
ROOT    = "./data"
dataset = Planetoid(root=ROOT, name="Cora")
data    = dataset[0]

# ── Same 80/10/10 split as Pipeline 1 ────────────────────────
torch.manual_seed(SEED)   # re-seed so RandomLinkSplit gives same split
transform = RandomLinkSplit(
    num_val                    = 0.10,
    num_test                   = 0.10,
    is_undirected              = True,
    add_negative_train_samples = True,
    neg_sampling_ratio         = 1.0,
)
train_data, val_data, test_data = transform(data)

print(f"Cora  |  nodes={data.num_nodes}  edges={data.num_edges//2}")
print(f"Train pos edges : {train_data.edge_label_index.shape[1]//2}")
print(f"Val   pos edges : {val_data.edge_label_index.shape[1]//2}")
print(f"Test  pos edges : {test_data.edge_label_index.shape[1]//2}")



Cora  |  nodes=2708  edges=5278
Train pos edges : 4224
Val   pos edges : 527
Test  pos edges : 527


In [16]:
# ── CELL 4 ── Build node text representations ────────────────
#
#  The paper uses raw text (title + abstract) from He et al.'s
#  TAPE version of Cora.  Since that version requires a separate
#  download, we construct synthetic text from the BOW features
#  and node indices here.  This is a valid drop-in for testing
#  the pipeline end-to-end.
#
#  To use the real TAPE text:
#    1. Download from https://huggingface.co/datasets/hm3/tape_cora
#    2. Replace the `node_texts` list below with the actual strings.

# ── Option A: Synthetic text (works out of the box) ──────────
def bow_to_text(node_idx, feature_vec, top_k=20):
    active = np.where(feature_vec > 0)[0]

    if len(active) == 0:
        return f"This research paper has no clear topic."

    words = " ".join([f"machine learning topic {i}" for i in active[:top_k]])

    return f"This paper discusses topics in machine learning such as {words}."

features     = data.x.numpy()                    # (N, 1433)
node_texts   = [
    bow_to_text(i, features[i])
    for i in range(data.num_nodes)
]

print(f"✅ Node texts ready  |  total = {len(node_texts)}")
print(f"   Example: '{node_texts[0]}'")

# ── Option B: Real TAPE text (uncomment when available) ───────
# import json
# with open("tape_cora_texts.json") as f:
#     node_texts = json.load(f)   # list of strings, one per node
# print(f"✅ Loaded TAPE texts  |  {len(node_texts)} nodes")

✅ Node texts ready  |  total = 2708
   Example: 'This paper discusses topics in machine learning such as machine learning topic 19 machine learning topic 81 machine learning topic 146 machine learning topic 315 machine learning topic 774 machine learning topic 877 machine learning topic 1194 machine learning topic 1247 machine learning topic 1274.'


In [17]:
# ── CELL 5 ── Build triplet dataset for contrastive learning ─
#
#  For every anchor node v_i:
#    Positive v_j  : directly connected to v_i (existing edge)
#    Negative v_k  : unreachable or ≥ 3 hops from v_i
#
#  Negative sampling strategy (Section 3.2):
#    1. Rank nodes by degree (high-degree hubs prioritised)
#    2. Alternating random + BFS-guided sampling
#    3. Max 5 neg pairs per central node

def build_triplet_dataset(edge_index, num_nodes, node_texts,
                           max_neg_per_node=5, seed=SEED):
    """
    Returns list of (anchor_idx, positive_idx, negative_idx) tuples.
    """
    rng = random.Random(seed)
    np_rng = np.random.default_rng(seed)

    # Build adjacency list (undirected)
    adj = {i: set() for i in range(num_nodes)}
    src, dst = edge_index.cpu().numpy()
    for u, v in zip(src, dst):
        adj[u].add(v)
        adj[v].add(u)

    # Degree-ranked candidate pool for negatives
    degrees      = np.array([len(adj[i]) for i in range(num_nodes)])
    ranked_nodes = np.argsort(-degrees).tolist()   # high-degree first

    triplets = []

    for anchor in range(num_nodes):
        pos_neighbors = list(adj[anchor])
        if not pos_neighbors:
            continue

        neg_count  = 0
        use_random = True   # alternating flag

        for candidate in ranked_nodes:
            if neg_count >= max_neg_per_node:
                break
            if candidate == anchor:
                continue
            # Check: candidate must be ≥ 3 hops or unreachable
            # (BFS up to depth 2 from anchor)
            visited = {anchor}
            frontier = {anchor}
            for _ in range(2):
                next_f = set()
                for node in frontier:
                    next_f |= adj[node]
                frontier = next_f - visited
                visited |= frontier
            if candidate in visited:
                continue   # too close (≤ 2 hops) → skip

            # Sample a positive partner for this anchor
            pos = rng.choice(pos_neighbors)

            if use_random:
                # random sampling for diversity
                triplets.append((anchor, pos, candidate))
            else:
                # BFS-guided: pick a 3-hop reachable node if exists
                # (already guaranteed by the visited check above)
                triplets.append((anchor, pos, candidate))

            use_random = not use_random
            neg_count += 1

    rng.shuffle(triplets)
    print(f"  Triplet dataset: {len(triplets):,} samples  "
          f"({len(triplets)//num_nodes:.1f} avg per node)")
    return triplets


print("Building triplet dataset…")
triplets = build_triplet_dataset(
    train_data.edge_index,
    data.num_nodes,
    node_texts,
    max_neg_per_node=5,
)


Building triplet dataset…
  Triplet dataset: 12,940 samples  (4.0 avg per node)


In [ ]:
# ── CELL 6 ── PyTorch Dataset & DataLoader ───────────────────

class TripletTextDataset(Dataset):
    """
    Returns tokenised (anchor, positive, negative) text triples.
    """
    def __init__(self, triplets, node_texts, tokenizer, max_length=128):
        self.triplets   = triplets
        self.texts      = node_texts
        self.tokenizer  = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.triplets)

    def _encode(self, text):
        return self.tokenizer(
            text,
            max_length      = self.max_length,
            padding         = "max_length",
            truncation      = True,
            return_tensors  = "pt",
        )

    def __getitem__(self, idx):
        a, p, n = self.triplets[idx]
        enc_a = self._encode(self.texts[a])
        enc_p = self._encode(self.texts[p])
        enc_n = self._encode(self.texts[n])
        return {
            "anchor_ids":      enc_a["input_ids"].squeeze(0),
            "anchor_mask":     enc_a["attention_mask"].squeeze(0),
            "positive_ids":    enc_p["input_ids"].squeeze(0),
            "positive_mask":   enc_p["attention_mask"].squeeze(0),
            "negative_ids":    enc_n["input_ids"].squeeze(0),
            "negative_mask":   enc_n["attention_mask"].squeeze(0),
            "anchor_idx":      torch.tensor(a, dtype=torch.long),
            "positive_idx":    torch.tensor(p, dtype=torch.long),
            "negative_idx":    torch.tensor(n, dtype=torch.long),
        }


print("Loading RoBERTa tokenizer…")
ROBERTA_MODEL = "roberta-base"
tokenizer = RobertaTokenizer.from_pretrained(ROBERTA_MODEL)

# Split triplets 90/10 for train/val within Pipeline 2
split_idx    = int(0.9 * len(triplets))
train_trips  = triplets[:split_idx]
val_trips    = triplets[split_idx:]

BATCH_SIZE_LM = 16   # paper uses batch size 16 for LM fine-tuning

train_lm_dataset = TripletTextDataset(train_trips, node_texts, tokenizer)
val_lm_dataset   = TripletTextDataset(val_trips,   node_texts, tokenizer)

train_lm_loader  = DataLoader(train_lm_dataset, batch_size=BATCH_SIZE_LM,
                               shuffle=True,  num_workers=2, pin_memory=True)
val_lm_loader    = DataLoader(val_lm_dataset,  batch_size=BATCH_SIZE_LM,
                               shuffle=False, num_workers=2, pin_memory=True)

print(f"✅ DataLoaders ready")
print(f"   Train batches : {len(train_lm_loader)}")
print(f"   Val   batches : {len(val_lm_loader)}")



In [ ]:
# ── CELL 7 ── Feature Encoder: RoBERTa + Contrastive Head ────
#
#  Architecture:
#    1. RoBERTa-base encodes each node's text → [CLS] vector (768-d)
#    2. A 2-layer projection head reduces to 128-d  (matching H_S dim)
#    3. Contrastive loss (eq. 8) fine-tunes both
#
#  Only the classification / projection layer is unfrozen initially;
#  the top 2 transformer layers are also unfrozen for fine-tuning
#  (standard practice for graph-text contrastive learning).

class FeatureEncoder(nn.Module):
    """
    RoBERTa-base with a projection head.
    Outputs L2-normalised 128-d embeddings for contrastive loss.
    """
    def __init__(self, roberta_model_name, proj_dim=128, dropout=0.1):
        super().__init__()
        self.roberta   = RobertaModel.from_pretrained(roberta_model_name)
        hidden_size    = self.roberta.config.hidden_size   # 768

        # Projection head: 768 → 256 → 128
        self.proj = nn.Sequential(
            nn.Linear(hidden_size, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, proj_dim),
        )

        # Freeze all RoBERTa layers except the last 2 transformer blocks
        self._freeze_roberta_layers(unfreeze_last_n=2)

    def _freeze_roberta_layers(self, unfreeze_last_n=2):
        """Freeze all layers; unfreeze the last N transformer layers."""
        for param in self.roberta.parameters():
            param.requires_grad = False
        # Unfreeze last N encoder layers
        total_layers = len(self.roberta.encoder.layer)
        for i in range(total_layers - unfreeze_last_n, total_layers):
            for param in self.roberta.encoder.layer[i].parameters():
                param.requires_grad = True
        # Always unfreeze the pooler
        for param in self.roberta.pooler.parameters():
            param.requires_grad = True

    def encode(self, input_ids, attention_mask):
        """
        Returns L2-normalised 128-d embedding for a batch of texts.
        """
        out    = self.roberta(input_ids=input_ids,
                              attention_mask=attention_mask)
        cls    = out.last_hidden_state[:, 0, :]   # [CLS] token  (B, 768)
        proj   = self.proj(cls)                   # (B, 128)
        return F.normalize(proj, dim=-1)           # L2-normalised

    def forward(self, anchor_ids, anchor_mask,
                      positive_ids, positive_mask,
                      negative_ids, negative_mask):
        """
        Forward pass for triplet contrastive training.
        Returns (emb_anchor, emb_positive, emb_negative).
        """
        e_a = self.encode(anchor_ids,   anchor_mask)
        e_p = self.encode(positive_ids, positive_mask)
        e_n = self.encode(negative_ids, negative_mask)
        return e_a, e_p, e_n


print("Loading RoBERTa-base (this downloads ~500 MB on first run)…")
feature_encoder = FeatureEncoder(
    roberta_model_name = ROBERTA_MODEL,
    proj_dim           = 128,
    dropout            = 0.1,
).to(DEVICE)

trainable = sum(p.numel() for p in feature_encoder.parameters()
                if p.requires_grad)
total     = sum(p.numel() for p in feature_encoder.parameters())
print(f"✅ FeatureEncoder ready")
print(f"   Trainable params : {trainable:,} / {total:,}  "
      f"({100*trainable/total:.1f}%)")



In [20]:
# ── CELL 8 ── Contrastive Loss (eq. 8 from paper) ───────────
#
#  The paper defines a triplet-style loss using cosine similarity:
#
#  L_F = - (1/|Ω|) Σ y·log[ exp(d(x_i,x_j)) /
#                            (exp(d(x_i,x_j)) + exp(d(x_i,x_k))) ]
#        + (1/|ω|) Σ (1-y)·log[ exp(d(x_k,x_j)) /
#                                (exp(d(x_i,x_j)) + exp(d(x_k,x_j))) ]
#        + λ·l(θ)    ← L2 regularisation (handled by weight_decay)
#
#  d(·,·) = cosine similarity (embeddings already L2-normalised,
#            so dot product == cosine similarity).

class TripletContrastiveLoss(nn.Module):
    def __init__(self, temperature=0.07):
        super().__init__()
        self.tau = temperature

    def forward(self, e_a, e_p, e_n):
        sim_ap = (e_a * e_p).sum(dim=-1) / self.tau
        sim_an = (e_a * e_n).sum(dim=-1) / self.tau

        logits = torch.stack([sim_ap, sim_an], dim=1)
        labels = torch.zeros(len(sim_ap), dtype=torch.long).to(e_a.device)

        loss = F.cross_entropy(logits, labels)

        return loss, sim_ap.mean().detach(), sim_an.mean().detach()
criterion = TripletContrastiveLoss(temperature=0.07)
print("✅ Contrastive loss ready  |  temperature = 0.07")




✅ Contrastive loss ready  |  temperature = 0.07


In [21]:
# ── CELL 9 ── Training Pipeline 2 ────────────────────────────
#
#  Optimiser : Adam, lr=1e-3, weight_decay=1e-5
#  Scheduler : StepLR – decay factor 0.1 every 3 epochs
#  Early stop: patience=50 on val loss
#  Epochs    : max 20 (LM fine-tuning converges fast)

LR_LM        = 1e-3
WEIGHT_DECAY = 1e-5
MAX_EPOCHS   = 20
PATIENCE     = 5          # epochs without val improvement
DECAY_EVERY  = 3          # LR decay period (paper: decay 0.1 every 3 epochs)
DECAY_FACTOR = 0.1

optimizer_lm = Adam([
    {"params": feature_encoder.roberta.parameters(), "lr": 2e-5},
    {"params": feature_encoder.proj.parameters(),    "lr": 1e-3}
])
scheduler_lm = StepLR(optimizer_lm, step_size=DECAY_EVERY,
                       gamma=DECAY_FACTOR)

def run_epoch(loader, encoder, loss_fn, optimizer=None, device=DEVICE):
    """One train or eval pass. Returns avg loss."""
    is_train = optimizer is not None
    encoder.train() if is_train else encoder.eval()

    total_loss = 0.0
    context    = torch.enable_grad() if is_train else torch.no_grad()

    with context:
        for batch in loader:
            a_ids  = batch["anchor_ids"].to(device)
            a_mask = batch["anchor_mask"].to(device)
            p_ids  = batch["positive_ids"].to(device)
            p_mask = batch["positive_mask"].to(device)
            n_ids  = batch["negative_ids"].to(device)
            n_mask = batch["negative_mask"].to(device)

            e_a, e_p, e_n = encoder(a_ids, a_mask, p_ids, p_mask,
                                     n_ids, n_mask)
            loss, _, _ = loss_fn(e_a, e_p, e_n)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(
                    filter(lambda p: p.requires_grad, encoder.parameters()),
                    max_norm=1.0
                )
                optimizer.step()

            total_loss += loss.item()

    return total_loss / max(len(loader), 1)


print(f"\nTraining Pipeline 2  (max {MAX_EPOCHS} epochs)…\n")

best_val_loss  = float("inf")
patience_cnt   = 0
best_lm_ckpt   = None

for epoch in range(1, MAX_EPOCHS + 1):

    train_loss = run_epoch(train_lm_loader, feature_encoder,
                           criterion, optimizer_lm, DEVICE)
    val_loss   = run_epoch(val_lm_loader,   feature_encoder,
                           criterion, None,          DEVICE)
    scheduler_lm.step()

    current_lr = scheduler_lm.get_last_lr()[0]
    print(f"  Epoch {epoch:3d} | train={train_loss:.4f} | "
          f"val={val_loss:.4f} | lr={current_lr:.2e}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_cnt  = 0
        best_lm_ckpt  = {k: v.clone()
                         for k, v in feature_encoder.state_dict().items()}
    else:
        patience_cnt += 1
        if patience_cnt >= PATIENCE:
            print(f"\n  Early stop at epoch {epoch}  "
                  f"(best val loss = {best_val_loss:.4f})")
            break

# Reload best checkpoint
if best_lm_ckpt:
    feature_encoder.load_state_dict(best_lm_ckpt)

print("\n✅ Pipeline 2 training complete")


Training Pipeline 2  (max 20 epochs)…

  Epoch   1 | train=0.2826 | val=0.1307 | lr=2.00e-05
  Epoch   2 | train=0.1112 | val=0.0979 | lr=2.00e-05
  Epoch   3 | train=0.0831 | val=0.0502 | lr=2.00e-06
  Epoch   4 | train=0.0516 | val=0.0595 | lr=2.00e-06
  Epoch   5 | train=0.0498 | val=0.0564 | lr=2.00e-06
  Epoch   6 | train=0.0467 | val=0.0598 | lr=2.00e-07
  Epoch   7 | train=0.0467 | val=0.0608 | lr=2.00e-07
  Epoch   8 | train=0.0442 | val=0.0599 | lr=2.00e-07

  Early stop at epoch 8  (best val loss = 0.0502)

✅ Pipeline 2 training complete


In [22]:



# ── CELL 10 ── Extract full-graph node feature embeddings H_F ─
#
#  We run all N nodes through the fine-tuned encoder in batches
#  to produce H_F : (N, 128)  – the feature-only embedding matrix.

INFER_BATCH = 64

def extract_all_embeddings(encoder, texts, tokenizer,
                            max_length=128, batch_size=INFER_BATCH,
                            device=DEVICE):
    """
    Encode every node's text with the fine-tuned FeatureEncoder.
    Returns (N, 128) tensor on CPU.
    """
    encoder.eval()
    all_embs = []

    with torch.no_grad():
        for start in range(0, len(texts), batch_size):
            batch_texts = texts[start : start + batch_size]
            enc = tokenizer(
                batch_texts,
                max_length     = max_length,
                padding        = "max_length",
                truncation     = True,
                return_tensors = "pt",
            )
            ids  = enc["input_ids"].to(device)
            mask = enc["attention_mask"].to(device)
            emb  = encoder.encode(ids, mask)   # (B, 128) L2-normalised
            all_embs.append(emb.cpu())

    return torch.cat(all_embs, dim=0)   # (N, 128)


print("Extracting H_F for all nodes…")
H_F = extract_all_embeddings(feature_encoder, node_texts, tokenizer)
print(f"✅ H_F shape: {H_F.shape}  |  dtype: {H_F.dtype}")

Extracting H_F for all nodes…
✅ H_F shape: torch.Size([2708, 128])  |  dtype: torch.float32


In [23]:



# ── CELL 11 ── Evaluate link prediction with H_F alone ───────
#
#  Link score = cosine similarity between the two endpoint embeddings.
#  This mirrors the paper's ablation "Feat-Only Emb." baseline.

def evaluate_cosine(H, split_data):
    """
    Score each edge pair by cosine similarity of endpoint embeddings.
    Since H is already L2-normalised, cosine sim = dot product.
    """
    pairs  = split_data.edge_label_index   # (2, 2P)
    labels = split_data.edge_label.numpy()

    src_emb = H[pairs[0]].numpy()
    dst_emb = H[pairs[1]].numpy()
    scores  = (src_emb * dst_emb).sum(axis=-1)   # dot = cosine (normalised)

    ap  = average_precision_score(labels, scores)
    auc = roc_auc_score(labels, scores)
    return ap, auc

val_ap,  val_auc  = evaluate_cosine(H_F, val_data)
test_ap, test_auc = evaluate_cosine(H_F, test_data)

print(f"\n{'='*52}")
print(f"  Pipeline 2 (Feature-Only) – Evaluation")
print(f"{'='*52}")
print(f"  Val  AP={val_ap*100:.2f}%   AUC={val_auc*100:.2f}%")
print(f"  Test AP={test_ap*100:.2f}%   AUC={test_auc*100:.2f}%")
print(f"{'='*52}")


  Pipeline 2 (Feature-Only) – Evaluation
  Val  AP=51.26%   AUC=49.30%
  Test AP=53.25%   AUC=53.31%


In [24]:
# ── CELL 12 ── Sanity checks ──────────────────────────────────

print("\n── Sanity checks ──────────────────────────────────────")
print(f"  H_F min/max/mean : "
      f"{H_F.min():.4f} / {H_F.max():.4f} / {H_F.mean():.4f}")
print(f"  H_F NaN          : {torch.isnan(H_F).any().item()}")
print(f"  H_F Inf          : {torch.isinf(H_F).any().item()}")
print(f"  H_F norm (row 0) : {H_F[0].norm().item():.4f}  (should be ~1.0)")

# Check that connected nodes have higher cosine sim than disconnected
pos_mask  = test_data.edge_label.bool()
neg_mask  = ~pos_mask
pos_pairs = test_data.edge_label_index[:, pos_mask]
neg_pairs = test_data.edge_label_index[:, neg_mask]

sim_pos = (H_F[pos_pairs[0]] * H_F[pos_pairs[1]]).sum(dim=-1).mean().item()
sim_neg = (H_F[neg_pairs[0]] * H_F[neg_pairs[1]]).sum(dim=-1).mean().item()
print(f"\n  Avg cosine sim – positive pairs : {sim_pos:.4f}")
print(f"  Avg cosine sim – negative pairs : {sim_neg:.4f}")
print(f"  Δ = {sim_pos - sim_neg:.4f}  (positive → good separation)")





── Sanity checks ──────────────────────────────────────
  H_F min/max/mean : -0.3162 / 0.2951 / -0.0044
  H_F NaN          : False
  H_F Inf          : False
  H_F norm (row 0) : 1.0000  (should be ~1.0)

  Avg cosine sim – positive pairs : 0.9300
  Avg cosine sim – negative pairs : 0.9724
  Δ = -0.0424  (positive → good separation)


In [25]:
# ── CELL 13 ── Save H_F and model checkpoint ─────────────────

torch.save({
    "H_F":                 H_F,
    "feature_encoder":     feature_encoder.state_dict(),
    "tokenizer_name":      ROBERTA_MODEL,
    "node_texts":          node_texts,
    "train_data":          train_data,
    "val_data":            val_data,
    "test_data":           test_data,
    "num_nodes":           data.num_nodes,
}, "pipeline2_outputs.pt")

print("✅ H_F saved to pipeline2_outputs.pt")
print()
print("="*52)
print("  Pipeline 2 complete. Next step → Pipeline 3")
print("  (Graph Data Distribution Analysis via GraphGPT)")
print("="*52)

✅ H_F saved to pipeline2_outputs.pt

  Pipeline 2 complete. Next step → Pipeline 3
  (Graph Data Distribution Analysis via GraphGPT)
